In [1]:
import pandas as pd
df_all_shuffled_training_mass = pd.read_csv("event_candidate_dataset_training.csv")
print(df_all_shuffled_training_mass.head())

   EvID  N Electron candidates  N Muon candidates  N Pion candidates  \
0     0                      0                  0                  0   
1     1                      0                  1                  0   
2     2                      0                  0                  0   
3     3                      0                  1                  1   
4     4                      0                  1                  1   

   N Kaon candidates  N Proton candidates  N Tracks  Invariant Mass  
0                  0                    1         2        3.050309  
1                  0                    1         2        3.126002  
2                  1                    0         2        3.094285  
3                  1                    0         2        3.049432  
4                  1                    0         2        3.108283  


In [2]:
df_all_shuffled_training_mass.info()

<class 'pandas.DataFrame'>
RangeIndex: 67278 entries, 0 to 67277
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   EvID                   67278 non-null  int64  
 1   N Electron candidates  67278 non-null  int64  
 2   N Muon candidates      67278 non-null  int64  
 3   N Pion candidates      67278 non-null  int64  
 4   N Kaon candidates      67278 non-null  int64  
 5   N Proton candidates    67278 non-null  int64  
 6   N Tracks               67278 non-null  int64  
 7   Invariant Mass         67278 non-null  float64
dtypes: float64(1), int64(7)
memory usage: 4.1 MB


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
#------------------------------------------
# 1. Load and Prepare the Data
#------------------------------------------

X=df_all_shuffled_training_mass[['N Electron candidates', 'N Muon candidates', 'N Pion candidates', 'N Kaon candidates', 'N Proton candidates', 'N Tracks']].values
# Extract mass separately
# mass = df['Invariant Mass'].values

mass = df_all_shuffled_training_mass['Invariant Mass'].values
# Create a train/test split

X_train, X_test, mass_train, mass_test = train_test_split(X, mass, test_size=0.05, random_state=42)

I0000 00:00:1779716838.241246   22125 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779716838.278811   22125 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779716839.057375   22125 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
from sklearn.preprocessing import MinMaxScaler
#Initialize scaler

scaler = MinMaxScaler()
#Fit and transform training data, transform test data

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
print(X_train.shape, X_test.shape)

(63914, 6) (3364, 6)


In [6]:
import tensorflow as tf
print("TensorFlow version:", tf.version)
print("Available devices:", tf.config.list_physical_devices())

TensorFlow version: <module 'tensorflow._api.v2.version' from '/home/student/tbml/.venv/lib/python3.12/site-packages/tensorflow/_api/v2/version/__init__.py'>
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


E0000 00:00:1779716839.484826   22125 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Disable GPU (if any)
#------------------------------------------
# 2. Define the Autoencoder Model
#------------------------------------------

input_dim = X_train.shape[1] # should be 6
encoding_dim = 4 # latent space dimension
# Input layer

input_layer = Input(shape=(input_dim,))
# Encoding layer (compression)

encoded = Dense(encoding_dim, activation='relu')(input_layer)
# Decoding layer (reconstruction)

decoded = Dense(input_dim, activation='sigmoid')(encoded)
# Autoencoder model

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')

In [8]:
from tensorflow.keras.optimizers import Adam
autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse') # best, 185 epochs automatic stop

In [9]:
#------------------------------------------
#3. Train the Autoencoder
#------------------------------------------

early_stopping = EarlyStopping(
monitor='val_loss',
patience=2,
restore_best_weights=True
)

history = autoencoder.fit(X_train, X_train,
epochs=500,
# epochs=500,
batch_size=256,
# batch_size=64,
shuffle=True,
validation_data=(X_test, X_test),
verbose=2,
callbacks=[early_stopping]
)
with tf.device('/CPU:0'):
    history = autoencoder.fit(X_train, X_train,
    epochs=500,
    batch_size=64,
    validation_data=(X_test, X_test),
    callbacks=[early_stopping],
    verbose=2)

Epoch 1/500
250/250 - 1s - 2ms/step - loss: 0.1839 - val_loss: 0.1461
Epoch 2/500
250/250 - 0s - 780us/step - loss: 0.1049 - val_loss: 0.0714
Epoch 3/500
250/250 - 0s - 955us/step - loss: 0.0560 - val_loss: 0.0442
Epoch 4/500
250/250 - 0s - 755us/step - loss: 0.0373 - val_loss: 0.0317
Epoch 5/500
250/250 - 0s - 1ms/step - loss: 0.0284 - val_loss: 0.0254
Epoch 6/500
250/250 - 0s - 1ms/step - loss: 0.0238 - val_loss: 0.0220
Epoch 7/500
250/250 - 0s - 1ms/step - loss: 0.0209 - val_loss: 0.0196
Epoch 8/500
250/250 - 0s - 803us/step - loss: 0.0186 - val_loss: 0.0174
Epoch 9/500
250/250 - 0s - 712us/step - loss: 0.0163 - val_loss: 0.0150
Epoch 10/500
250/250 - 0s - 747us/step - loss: 0.0109 - val_loss: 0.0083
Epoch 11/500
250/250 - 0s - 774us/step - loss: 0.0070 - val_loss: 0.0063
Epoch 12/500
250/250 - 0s - 976us/step - loss: 0.0058 - val_loss: 0.0055
Epoch 13/500
250/250 - 0s - 955us/step - loss: 0.0052 - val_loss: 0.0051
Epoch 14/500
250/250 - 0s - 775us/step - loss: 0.0048 - val_loss: 0.

In [10]:
print(f"Training stopped at epoch {len(history.history['loss'])}")
print(f"Training stopped at epoch {len(history.history['loss'])}")

Training stopped at epoch 7
Training stopped at epoch 7
